In [24]:
import pandas as pd
import numpy as np
from sklearn.cluster import OPTICS
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns
from statistics import mode
from sklearn.metrics import recall_score
from sklearn.ensemble import RandomForestClassifier

# seleccion data frame
X_train = pd.read_csv(r"C:\Users\mejia\Desktop\Estudio\IA\Proyecto4-IA\Datos\TrainX.csv")
Y_train = pd.read_csv(r"C:\Users\mejia\Desktop\Estudio\IA\Proyecto4-IA\Datos\TrainY.csv")
X_test = pd.read_csv(r"C:\Users\mejia\Desktop\Estudio\IA\Proyecto4-IA\Datos\TestX.csv")
Y_test = pd.read_csv(r"C:\Users\mejia\Desktop\Estudio\IA\Proyecto4-IA\Datos\TestY.csv")
X_val = pd.read_csv(r"C:\Users\mejia\Desktop\Estudio\IA\Proyecto4-IA\Datos\ValidationX.csv")
Y_val = pd.read_csv(r"C:\Users\mejia\Desktop\Estudio\IA\Proyecto4-IA\Datos\ValidationY.csv")


#### No hace falta limpiar los datos, transformar, ni normalizar puesto que este proceso se hizo con el dataset original

#### No es necesario conseguir coeficiente de silueta/codo para OPTICS

# a. Tomar minimo 3 hiperparametros

# b. Sacar grupos, con parametros diferentes
#### Cambiando 3 veces el valor de cada hiperparámetro y aplicar el algoritmo varias veces (varios for's anidados para cambiar los hiperparámetros)

In [25]:
eps_values = [5, 10, 15]
min_samples_values = [5, 10, 15]
min_clusters_values = [10, 20, 30]

# ===============================
# Lista de resultados
# ===============================
resultados = []

# ===============================
# Repetir 3 veces por combinación
# ===============================
for eps in eps_values:
    for min_samples in min_samples_values:
        for mcs in min_clusters_values:
            for rep in range(3):  # repetir 3 veces cada combinación
                model = OPTICS(max_eps=eps,
                               min_samples=min_samples,
                               min_cluster_size=mcs)
                model.fit(X_train)
                labels_train = model.labels_

                resultados.append({
                    'eps': eps,
                    'min_samples': min_samples,
                    'min_cluster_size': mcs,
                    'repeticion': rep + 1
                })


KeyboardInterrupt: 

In [ ]:
from sklearn.cluster import OPTICS
from sklearn.metrics import recall_score
from scipy.stats import mode
import numpy as np
import pandas as pd

# ===============================
# Función auxiliar para calcular recal
# ===============================
def calcular_recall(y_true, labels_pred):
    etiquetas_map = {}
    for cluster in np.unique(labels_pred):
        if cluster == -1:
            continue
        mask = labels_pred == cluster
        cluster_mode = mode(y_true[mask])
        etiquetas_map[cluster] = cluster_mode.mode[0] if hasattr(cluster_mode, "mode") else cluster_mode[0]
    y_pred_mapeado = np.array([etiquetas_map.get(c, -1) for c in labels_pred])
    return recall_score(y_true, y_pred_mapeado, average='macro', zero_division=0)

# ===============================
# Evaluar cada combinación
# ===============================
resultados = []

for eps in eps_values:
    for min_samples in min_samples_values:
        for mcs in min_clusters_values:
            # Entrenar en train
            model = OPTICS(max_eps=eps,
                           min_samples=min_samples,
                           min_cluster_size=mcs)
            model.fit(X_train)
            labels_train = model.labels_

            # Entrenar en val
            model_val = OPTICS(max_eps=eps,
                               min_samples=min_samples,
                               min_cluster_size=mcs)
            model_val.fit(X_val)
            labels_val = model_val.labels_

            # Calcular Recall
            recall_train = calcular_recall(Y_train, labels_train)
            recall_val = calcular_recall(Y_val, labels_val)

            # Calcular error = 1 - recall
            error_train = 1 - recall_train
            error_val = 1 - recall_val

            resultados.append({
                'eps': eps,
                'min_samples': min_samples,
                'min_cluster_size': mcs,
                'Error_Train': error_train,
                'Error_Val': error_val
            })

# ===============================
# Tabla comparativa
# ===============================
df_resultados = pd.DataFrame(resultados)
print("Tabla comparativa de hiperparámetros y errores (basado en Recall):\n")
print(df_resultados)


Tabla comparativa de hiperparámetros y errores (basado en Recall):

    eps  min_samples  min_cluster_size  Error_Train  Error_Val
0     5            5                10          0.5        0.5
1     5            5                20          0.5        0.5
2     5            5                30          0.5        0.5
3     5           10                10          0.5        0.5
4     5           10                20          0.5        0.5
5     5           10                30          0.5        0.5
6     5           15                10          0.5        0.5
7     5           15                20          0.5        0.5
8     5           15                30          0.5        0.5
9    10            5                10          0.5        0.5
10   10            5                20          0.5        0.5
11   10            5                30          0.5        0.5
12   10           10                10          0.5        0.5
13   10           10                20          0.

In [ ]:
# ===============================
# Tomar la mejor combinación (menor error_val)
# ===============================

best_params = df_resultados.sort_values(by='Error_Val', ascending=True).iloc[0]
print("\nMejores hiperparámetros encontrados:")
print(best_params)

# Entrenar con los mejores hiperparámetros en test
model_test = OPTICS(max_eps=best_params['eps'],
                    min_samples=int(best_params['min_samples']),
                    min_cluster_size=int(best_params['min_cluster_size']))
model_test.fit(X_test)
labels_test = model_test.labels_

recall_test = calcular_recall(Y_test, labels_test)
error_test = 1 - recall_test
print(f"\nError de Test: {error_test:.4f}")

# ===============================
# Predecir un dato nuevo (modelo ML)
# ===============================
# Entrenar un modelo supervisado (ej: RandomForest)
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, Y_train)

# Crear un dato inventado con la misma estructura de X
nuevo_dato = np.array([X_train[0]])  # puedes cambiar valores si quieres
prediccion = clf.predict(nuevo_dato)

print(f"\nPredicción del modelo ML para el nuevo dato: {prediccion[0]}")



Mejores hiperparámetros encontrados:
eps                  5.0
min_samples          5.0
min_cluster_size    10.0
Error_Train          0.5
Error_Val            0.5
Name: 0, dtype: float64


ValueError: Mix of label input types (string and number)

# c. 